In [17]:
import os
from dotenv import load_dotenv
from typing import Annotated, TypedDict, Sequence

from langgraph.prebuilt import create_react_agent
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import Tool

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer

from langchain_groq import ChatGroq

# --------------------------
# 1. Create Retriever Tool (Cricket)
# --------------------------

# Load cricket points table page
docs = WebBaseLoader("https://www.iplt20.com/matches/points-table").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Embeddings with sentence-transformers
class STEmbeddings:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

embedding = STEmbeddings()
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

def cricket_retriever_tool(__arg1: str) -> str:
    print("🏏 Using CricketRetriever tool")
    docs = retriever.invoke(__arg1)
    if not docs:
        return f"No cricket info found for: {__arg1}"
    return "\n".join([doc.page_content for doc in docs])

cricket_tool = Tool(
    name="CricketRetriever",
    description="Fetch IPL points table and cricket info",
    func=cricket_retriever_tool
)

print(cricket_tool.name)

# --------------------------
# 2. Initialize Groq LLM
# --------------------------
load_dotenv()
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# --------------------------
# 3. Define Agent Node
# --------------------------
tools = [cricket_tool]
react_node = create_react_agent(llm, tools)

# --------------------------
# 4. LangGraph Agent State
# --------------------------
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# --------------------------
# 5. Build LangGraph Graph
# --------------------------
builder = StateGraph(AgentState)
builder.add_node("react_agent", react_node)
builder.set_entry_point("react_agent")
builder.add_edge("react_agent", END)

graph = builder.compile()

# --------------------------
# 6. Run the ReAct Agent
# --------------------------
if __name__ == "__main__":
    user_query = "Show me the IPL points table and explain the top teams."
    state = {"messages": [HumanMessage(content=user_query)]}
    result = graph.invoke(state)

    print("\n✅ Final Answer:\n", result["messages"][-1].content)


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.
C:\Users\admin\AppData\Local\Temp\ipykernel_11756\726009519.py:70: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_node = create_react_agent(llm, tools)


CricketRetriever


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=CricketRetriever>{"__arg1": "IPL points table and top teams"}'}}

In [20]:
import os
from dotenv import load_dotenv
from typing import Annotated, TypedDict, Sequence

from langgraph.prebuilt import create_react_agent
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import Tool

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

# --------------------------
# 1. Create Retriever Tool (Cricket)
# --------------------------

# Load IPL points table page
docs = WebBaseLoader("https://www.iplt20.com/matches/points-table").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Use HuggingFace embeddings (clean, no custom class)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

def cricket_retriever_tool(query: str) -> str:
    print("🏏 Using CricketRetriever tool")
    docs = retriever.invoke(query)
    if not docs:
        return f"No cricket info found for: {query}"
    return "\n".join([doc.page_content for doc in docs])

cricket_tool = Tool(
    name="CricketRetriever",
    description="Use this tool to fetch IPL points table info",
    func=cricket_retriever_tool
)

print(cricket_tool.name)

# --------------------------
# 2. Initialize Groq LLM
# --------------------------
load_dotenv()
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# --------------------------
# 3. Define the Agent Node
# --------------------------
tools = [cricket_tool]
react_node = create_react_agent(llm, tools)

# --------------------------
# 4. LangGraph Agent State
# --------------------------
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# --------------------------
# 5. Build LangGraph Graph
# --------------------------
builder = StateGraph(AgentState)
builder.add_node("react_agent", react_node)
builder.set_entry_point("react_agent")
builder.add_edge("react_agent", END)

graph = builder.compile()

# --------------------------
# 6. Run the ReAct Agent
# --------------------------
if __name__ == "__main__":
    user_query = "Show me the IPL points table and explain the top teams."
    state = {"messages": [HumanMessage(content=user_query)]}
    result = graph.invoke(state)

    print("\n✅ Final Answer:\n", result["messages"][-1].content)

CricketRetriever


C:\Users\admin\AppData\Local\Temp\ipykernel_11756\1134080447.py:62: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_node = create_react_agent(llm, tools)


🏏 Using CricketRetriever tool

✅ Final Answer:
 The IPL points table is a ranking system that shows the performance of each team in the Indian Premier League. The top teams are determined by the number of points they have earned, which is calculated based on their wins, losses, and ties.

Here's a general explanation of the top teams in the IPL points table:

1. **Chennai Super Kings**: They are one of the most successful teams in the IPL, with three titles to their name. They have a strong squad and a good balance of experienced players and young talent.
2. **Mumbai Indians**: They are the most successful team in the IPL, with five titles. They have a strong squad and a good balance of experienced players and young talent.
3. **Gujarat Titans**: They are a relatively new team, but they have made a strong impact in their first few seasons. They have a good balance of experienced players and young talent.
4. **Lucknow Super Giants**: They are another relatively new team, but they have m